# Model Interpretability with SHAP

Explains ML model predictions using SHAP (SHapley Additive exPlanations).
Produces beeswarm, dependence, force, and waterfall plots for individual
predictions and global feature importance.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.models.supervised_ml import (
    build_features_and_target,
    train_random_forest,
    train_xgboost,
)
from src.models.interpret import (
    create_explainer,
    explain_model,
    plot_shap_summary,
    plot_shap_dependence,
    plot_shap_force,
)
from src.models.ensemble import build_stacking_ensemble

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
# ── Load data ─────────────────────────────────────────────────────
returns = pd.read_csv('../data/raw/stock_returns.csv', index_col='Date')
returns.index = pd.to_datetime(returns.index)
returns = returns.select_dtypes(include='number').dropna()

TARGET = 'AAPL'
print(f'Returns shape: {returns.shape}')

In [ ]:
# ── Build features ────────────────────────────────────────────────
X, y, feat_names = build_features_and_target(
    returns, ticker=TARGET, lags=5, target='direction'
)
print(f'Features: {X.shape[1]}, Samples: {X.shape[0]}')

## Train Model

In [ ]:
# Train XGBoost classifier
result = train_xgboost(X, y, task='classification')
print(f'Accuracy: {result["metrics"]["accuracy"]:.4f}')

model = result['model']
X_test = result['X_test']

## SHAP Explanation

In [ ]:
# Create explainer and compute SHAP values
explanation = explain_model(
    model,
    X_test,
    feature_names=feat_names,
    method='tree',
    output_dir='../reports/figures',
)

print(f'Expected value: {explanation["expected_value"]:.4f}')
print(f'SHAP values shape: {explanation["shap_values"].shape}')

In [ ]:
# Global summary (beeswarm) plot
fig = plot_shap_summary(
    explanation['shap_values'],
    X_test,
    feature_names=feat_names,
    max_display=15,
)
fig.savefig('../reports/figures/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Dependence plot for the most important feature
top_feature = feat_names[np.argmax(np.abs(explanation['shap_values']).mean(0))]
print(f'Top feature: {top_feature}')

fig = plot_shap_dependence(
    explanation['shap_values'],
    X_test,
    feature_idx=feat_names.index(top_feature),
    feature_names=feat_names,
)
fig.savefig(f'../reports/figures/shap_dependence_{top_feature}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Force plot for a single prediction
sample_idx = 0
fig = plot_shap_force(
    explanation['explainer'],
    X_test[sample_idx:sample_idx+1],
    feature_names=feat_names,
)
plt.show()